**Data Source Prep for Philosophy_Reviews.jsonl**


*   Download Amazon book & review metadata from mcauleylab.ucsd.edu
*   Filter by books in Phiosophy category metadata and join to the amazon review data.
*   Output: Philosophy_Reviews.jsonl

*  Books reviews: https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Books.jsonl.gz
Books metadata: https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Books.jsonl.gz









In [1]:
import gzip
import itertools
import json


In [2]:
# Cell 1 — metadata download
!wget -nc https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Books.jsonl.gz
!ls -lh meta_Books.jsonl.gz



--2026-08-18 01:19:57--  https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/meta_categories/meta_Books.jsonl.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4942125770 (4.6G) [application/gzip]
Saving to: ‘meta_Books.jsonl.gz’

meta_Books.jsonl.gz 100%[===================>]   4.60G  60.4MB/s    in 86s     

2026-08-18 01:21:24 (54.5 MB/s) - ‘meta_Books.jsonl.gz’ saved [4942125770/4942125770]

-rw-r--r-- 1 root root 4.7G Jan 16  2025 meta_Books.jsonl.gz


In [3]:
def is_philosophy(record):
    cats = record.get("categories") or []      # None and missing both become []
    return any("philosoph" in c.lower() for c in cats)

In [4]:
matching_asins = set()
asin_to_info = {}          # was: asin_to_title = {}

n_total = 0
n_kept = 0

with gzip.open("meta_Books.jsonl.gz", "rt") as f:
  for line in f:
    n_total += 1
    record = json.loads(line)

    if is_philosophy(record):
      asin = record["parent_asin"]
      matching_asins.add(asin)
      asin_to_info[asin] = {
            "title": record.get("title", ""),
            "author": (record.get("author") or {}).get("name"),
            "categories": record.get("categories") or [],
            "rating_number": record.get("rating_number", 0),
            "average_rating": record.get("average_rating"),
        }
      n_kept += 1
        # --------------------------------------------------------------

    if n_total % 50_000 == 0:        # heartbeat so you know it's alive
      print(f"{n_total:,} scanned, {n_kept:,} kept")

print(f"DONE: {n_total:,} scanned, {n_kept:,} kept")




50,000 scanned, 150 kept
100,000 scanned, 308 kept
150,000 scanned, 478 kept
200,000 scanned, 618 kept
250,000 scanned, 777 kept
300,000 scanned, 930 kept
350,000 scanned, 1,069 kept
400,000 scanned, 1,246 kept
450,000 scanned, 1,413 kept
500,000 scanned, 1,568 kept
550,000 scanned, 1,723 kept
600,000 scanned, 1,871 kept
650,000 scanned, 2,033 kept
700,000 scanned, 2,205 kept
750,000 scanned, 2,359 kept
800,000 scanned, 2,504 kept
850,000 scanned, 2,635 kept
900,000 scanned, 2,797 kept
950,000 scanned, 2,947 kept
1,000,000 scanned, 3,096 kept
1,050,000 scanned, 3,244 kept
1,100,000 scanned, 3,393 kept
1,150,000 scanned, 3,526 kept
1,200,000 scanned, 3,663 kept
1,250,000 scanned, 3,803 kept
1,300,000 scanned, 3,964 kept
1,350,000 scanned, 4,128 kept
1,400,000 scanned, 4,278 kept
1,450,000 scanned, 4,428 kept
1,500,000 scanned, 4,567 kept
1,550,000 scanned, 4,712 kept
1,600,000 scanned, 4,873 kept
1,650,000 scanned, 5,006 kept
1,700,000 scanned, 5,171 kept
1,750,000 scanned, 5,312 kept
1

In [5]:

print(len(matching_asins))
# peek at 20 titles — do they look like what the filter claims?
for info in itertools.islice(asin_to_info.values(), 20):
    print(" •", info["title"], info["categories"])

17137
 • Soulmaking: Uncommon Paths to Self-Understanding ['Books', 'Politics & Social Sciences', 'Philosophy']
 • Story of Philosophy ['Books', 'Politics & Social Sciences', 'Philosophy']
 • The Yoga Of Kashmir Shaivism:Consciousness Is Everything ['Books', 'Politics & Social Sciences', 'Philosophy']
 • Philosophy: 100 Essential Thinkers: The Ideas That Have Shaped Our World ['Books', 'Politics & Social Sciences', 'Philosophy']
 • Cosmopolitics I (Volume 9) (Posthumanities) ['Books', 'Science & Math', 'History & Philosophy']
 • Marx's Concept of Man (Continuum Impacts) ['Books', 'Politics & Social Sciences', 'Philosophy']
 • Find Your Rainbow - I Already Found Mine ['Books', 'Politics & Social Sciences', 'Philosophy']
 • Destiny, Freedom, and the Soul: What Is the Meaning of Life? (Osho Life Essentials) ['Books', 'Politics & Social Sciences', 'Philosophy']
 • The 3 Minutes Gratitude Journal: Start Your Day With Gratitude, Cultivate an Attitude of Awareness to Develop Mindfulness and G

In [6]:

found = 0
with gzip.open("meta_Books.jsonl.gz", "rt") as f:
    for line in f:
        r = json.loads(line)
        if is_philosophy(r):
            print(sorted(r.keys()))          # does 'author' appear at all?
            print("author:", r.get("author"))
            print("store:", r.get("store"))
            print("details:", r.get("details"))
            print("---")
            found += 1
            if found == 5:
                break

['author', 'average_rating', 'bought_together', 'categories', 'description', 'details', 'features', 'images', 'main_category', 'parent_asin', 'price', 'rating_number', 'store', 'subtitle', 'title', 'videos']
author: {'avatar': 'https://m.media-amazon.com/images/I/81YM5UC+sVL._SY600_.jpg', 'name': 'Michael Grosso', 'about': ["Michael Grosso, Ph.D, is an independent scholar, associated with an ongoing Seminar at Esalen on the role of mind in the cosmos. His latest book focuses on psychic anomalies that challenge reductive materialism. The emphasis is on waking up to the full girth of our potential. Michael Grosso,Ph.D., has taught humanities and philosophy at Marymount Manhattan College, City University of New York, and New Jersey City University. He is on the Board of Directors of the American Philosophical Practitioner's Association, and is a past editor of the journal for that association."]}
store: Michael Grosso (Author),  Raymond Moody Jr. (Foreword)
details: {'Publisher': 'Hampton

In [7]:

!wget -nc https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Books.jsonl.gz
!ls -lh Books.jsonl.gz

--2026-08-18 01:25:16--  https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Books.jsonl.gz
Resolving mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)... 137.110.161.5
Connecting to mcauleylab.ucsd.edu (mcauleylab.ucsd.edu)|137.110.161.5|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6216820644 (5.8G) [application/gzip]
Saving to: ‘Books.jsonl.gz’

Books.jsonl.gz      100%[===================>]   5.79G  18.1MB/s    in 3m 20s  

2026-08-18 01:28:36 (29.7 MB/s) - ‘Books.jsonl.gz’ saved [6216820644/6216820644]

-rw-r--r-- 1 root root 5.8G Jan 16  2025 Books.jsonl.gz


In [8]:
n_total = 0
n_kept = 0

with gzip.open("Books.jsonl.gz", "rt") as f_in, \
     open("philosophy_reviews.jsonl", "w") as f_out:
    for line in f_in:
        n_total += 1
        record = json.loads(line)
        if record["parent_asin"] in matching_asins:
            f_out.write(line)
            n_kept += 1
        if n_total % 1_000_000 == 0:
            print(f"{n_total:,} scanned, {n_kept:,} kept")

print(f"DONE: kept {n_kept:,} of {n_total:,}")

1,000,000 scanned, 1,333 kept
2,000,000 scanned, 2,626 kept
3,000,000 scanned, 4,507 kept
4,000,000 scanned, 5,773 kept
5,000,000 scanned, 7,164 kept
6,000,000 scanned, 8,405 kept
7,000,000 scanned, 9,770 kept
8,000,000 scanned, 11,361 kept
9,000,000 scanned, 12,942 kept
10,000,000 scanned, 14,651 kept
11,000,000 scanned, 16,161 kept
12,000,000 scanned, 17,589 kept
13,000,000 scanned, 19,275 kept
14,000,000 scanned, 20,516 kept
15,000,000 scanned, 21,930 kept
16,000,000 scanned, 23,509 kept
17,000,000 scanned, 25,587 kept
18,000,000 scanned, 27,280 kept
19,000,000 scanned, 28,739 kept
20,000,000 scanned, 30,568 kept
21,000,000 scanned, 32,648 kept
22,000,000 scanned, 34,824 kept
23,000,000 scanned, 37,079 kept
24,000,000 scanned, 39,531 kept
25,000,000 scanned, 41,907 kept
26,000,000 scanned, 44,396 kept
27,000,000 scanned, 47,013 kept
28,000,000 scanned, 49,555 kept
29,000,000 scanned, 52,342 kept
DONE: kept 53,518 of 29,475,453


In [13]:
import json

import pandas as pd

df = pd.read_json("philosophy_reviews.jsonl", lines=True)
print(f"{len(df):,} reviews, {df['parent_asin'].nunique():,} items, {df['user_id'].nunique():,} users")

# --- decide the floor with evidence, not vibes ---
item_counts = df["parent_asin"].value_counts()
for floor in (5, 10, 15, 20, 30):
    print(f"floor ≥{floor}: {(item_counts >= floor).sum():,} items survive")

# --- iterated k-core pruning: repeat both floors until stable ---
FLOOR = 3
df_pruned = df.copy()
prev_len = -1
passes = 0

while len(df_pruned) != prev_len:
    prev_len = len(df_pruned)
    passes += 1

    ic = df_pruned["parent_asin"].value_counts()
    df_pruned = df_pruned[df_pruned["parent_asin"].isin(ic[ic >= FLOOR].index)]

    uc = df_pruned["user_id"].value_counts()
    df_pruned = df_pruned[df_pruned["user_id"].isin(uc[uc >= 2].index)]

    print(f"pass {passes}: {len(df_pruned):,} reviews, "
          f"{df_pruned['parent_asin'].nunique():,} items, "
          f"{df_pruned['user_id'].nunique():,} users")

# invariant check — refuses to proceed if the floors aren't truly met
assert df_pruned["parent_asin"].value_counts().min() >= FLOOR
assert df_pruned["user_id"].value_counts().min() >= 2
print(f"stable after {passes} passes ✓")

df_pruned.to_json("philosophy_reviews_pruned.jsonl", orient="records", lines=True)

surviving = set(df_pruned["parent_asin"].unique())
catalog = {asin: info for asin, info in asin_to_info.items() if asin in surviving}

with open("catalog.json", "w") as f:
    json.dump(catalog, f)

print(f"{len(catalog):,} items in final catalog")

53,518 reviews, 17,112 items, 44,074 users
floor ≥5: 2,033 items survive
floor ≥10: 835 items survive
floor ≥15: 467 items survive
floor ≥20: 311 items survive
floor ≥30: 182 items survive
pass 1: 7,674 reviews, 2,562 items, 2,771 users
pass 2: 4,682 reviews, 939 items, 1,727 users
pass 3: 4,240 reviews, 769 items, 1,563 users
pass 4: 4,158 reviews, 738 items, 1,537 users
pass 5: 4,154 reviews, 736 items, 1,537 users
pass 6: 4,154 reviews, 736 items, 1,537 users
stable after 6 passes ✓
736 items in final catalog


In [12]:
print(df_pruned["parent_asin"].value_counts().min(),
      df_pruned["user_id"].value_counts().min())

1 2


In [14]:
import wandb

run = wandb.init(project="book-recommender", job_type="data-prep")

artifact = wandb.Artifact(
    name="philosophy-dataset",
    type="dataset",
    description="Amazon Reviews 2023 Books, filtered to philosophy catalog",
    metadata={
        "source": "McAuley Lab Amazon Reviews 2023, Books category",
        "filter": "any category path element contains 'philosoph' (case-insensitive)",
        "join_key": "parent_asin",
        "rating_floor": FLOOR,
        "n_items": len(catalog),
        "n_reviews": len(df_pruned),
        "n_users": df_pruned["user_id"].nunique(),
    },
)
artifact.add_file("philosophy_reviews_pruned.jsonl")   # the frozen training data
artifact.add_file("catalog.json")
run.log_artifact(artifact)
run.finish()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: malloryberg (malloryberg-university-of-denver) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
